In [ ]:
!gdown --id 1npjy1l8BwDe12KQ7MQrb7bo9jzkEWo5s
!unzip "card_fraud_detection.zip"


In [4]:
pip install imblearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd 
import numpy as np 
import math
import random

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
df = pd.read_csv("D:/Code/Machine-Learning/Data/Softmax Regression/creditcard.csv")
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [7]:
print(df.shape)

(284807, 31)


In [8]:
df.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [9]:
# Chuyển DataFrame sang mảng NumPy
dataset_arr = df.to_numpy()

# Tách đặc trưng (X) và nhãn (y)
X, y = dataset_arr[:, :-1].astype(np.float64), dataset_arr[:, -1].astype(np.uint8)
print(X.shape)
print(y.shape)

# Thêm thành phần bias vào ma trận đặc trưng
intercept = np.ones((X.shape[0], 1))
X_b = np.concatenate((intercept, X), axis=1)
print(X_b.shape)

# Mã hóa nhãn sang dạng one-hot
n_classes = np.unique(y, axis=0).shape[0]
n_samples = y.shape[0]

y_encoded = np.array([np.zeros(n_classes) for _ in range(n_samples)])
y_encoded[np.arange(n_samples), y] = 1


(284807, 30)
(284807,)
(284807, 31)


In [10]:
val_size = 0.2
test_size = 0.125
random_state = 2
is_shuffle = True

 # Chia tập huấn luyện và validation
X_train, X_val, y_train, y_val = train_test_split(
    X_b, y_encoded,
    test_size=val_size,
    random_state=random_state,
    shuffle=is_shuffle
)

# Chia tiếp một phần từ tập huấn luyện để làm tập kiểm thử (test)
X_train, X_test, y_train, y_test = train_test_split(
    X_train, y_train,
    test_size=test_size,
    random_state=random_state,
    shuffle=is_shuffle
)

# Chuẩn hoá đặc trưng (bỏ qua cột bias)
normalizer = StandardScaler()
X_train[:, 1:] = normalizer.fit_transform(X_train[:, 1:])
X_val[:, 1:] = normalizer.transform(X_val[:, 1:])
X_test[:, 1:] = normalizer.transform(X_test[:, 1:])

# Cân bằng dữ liệu huấn luyện bằng SMOTE
smote = SMOTE(random_state=random_state, sampling_strategy="minority")
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train[:, 1:], np.argmax(y_train, axis=1)
)

# Thêm lại thành phần bias sau khi resample
intercept_balanced = np.ones((X_train_balanced.shape[0], 1))
X_train = np.concatenate((intercept_balanced, X_train_balanced),axis=1)
# Mã hóa lại nhãn theo dạng one-hot
y_train = np.zeros((y_train_balanced.shape[0], n_classes))
y_train[np.arange(y_train_balanced.shape[0]), y_train_balanced] = 1

# Kiểm tra lại phân phối lớp
print(f"Balanced training samples: {X_train.shape}")
print(f"Balanced training samples: {y_train.shape}")
print(f"Class distribution: {np.bincount(y_train_balanced)}")

Balanced training samples: (398002, 31)
Balanced training samples: (398002, 2)
Class distribution: [199001 199001]


In [76]:
import math

def softmax(z):
    # z có kích thước [label, batch_size]
    label_count = len(z)
    batch_size = len(z[0])
    
    res = []
    for t in range(label_count):
        res.append([])
        
    for i in range(batch_size):
        # 1. Tìm max logit của mẫu dữ liệu thứ i để chống tràn số
        max_z = z[0][i]
        for t in range(1, label_count):
            if z[t][i] > max_z:
                max_z = z[t][i]
                
        # 2. Tính tổng e^(z - max_z) của TẤT CẢ các classes cho mẫu thứ i
        sum_exp = 0
        for t in range(label_count):
            sum_exp += math.e ** (z[t][i] - max_z)
            
        # 3. Tính phân phối xác suất cho từng class của mẫu thứ i
        for t in range(label_count):
            prob = (math.e ** (z[t][i] - max_z)) / sum_exp
            res[t].append(prob)
            
    # Chuyển vị ma trận về kích thước [batch_size, label] giống logic cũ của bạn
    res_T = [list(x) for x in zip(*res)]
    return res_T

In [71]:
def predict(batch_size, N, label, X, theta):
    # X = [[x1 x2 x31] [x1 x2 x31] ... [x1 x2 x31]]
    # theta = [x1 y1]
    # theta = [x2 y2]
    # ....
    # theta = [x31 y31]
    res = []
    for t in range(label): #2
        tmp = []
        for i in range(batch_size): # 1024
            product = 0
            for j in range(N): # 31
                product += X[i][j] * theta[j][t] 
            tmp.append(product)
        res.append(tmp)
    res = softmax(res)
    return res
    # return [[x1 x2 x_batch] [y1 y2 y_batch]]

In [ ]:
def compute_loss(label, N, y_hat, y_i):
    res = 0
    epsilon = 1e-15
    # print(label, N)
    for i in range(label): # 2
        for j in range(N): # 1024
            res += y_i[j][i] * math.log(y_hat[j][i] + epsilon)
    res = -1/N * res 
    return res

In [59]:
def compute_gradient(N, X_i, y_i, y_hat):
    dZ = (y_hat - y_i) / N
    dW = X_i.T @ dZ
    return dW

In [60]:
def update_theta(theta, gradient, lr):
    new_w = theta - lr * gradient
    return new_w

In [78]:
lr = 0.01
epochs = 30
batch_size = 1024
n_features = X_train.shape[1]
# print(X_train.shape)

np.random.seed(random_state)
theta = np.random.uniform(size=(n_features, n_classes))
# print(theta.shape)


train_accs = []
train_losses = []
val_accs = []
val_losses = []
for epoch in range(epochs):

    for i in range(epochs, X_train.shape[0], batch_size):
        X_i = X_train[i:i+batch_size]
        y_i = y_train[i:i+batch_size]
        y_hat = predict(X_i.shape[0], n_features, theta.shape[1], X_i, theta)
                        #1024           #31         #2
        train_loss = compute_loss(theta.shape[1], X_i.shape[0], y_hat, y_i)
        # print("loss: ", train_loss)
        gradient = compute_gradient(X_i.shape[0], X_i, y_i, y_hat)
        # print("gradient: ", gradient)
        # print("old theta: ", theta)
        theta = update_theta(theta, gradient, lr)
        # print("new theta: ", theta)
    

In [79]:
print(theta)

[[ 1.81878687 -1.35686573]
 [ 0.78734282  0.19764205]
 [ 0.18018838  0.57051425]
 [ 0.33452214  0.48939746]
 [ 0.33998111  0.22650084]
 [-0.05097623  1.20125216]
 [ 0.08253085  0.56562722]
 [ 0.69491559  0.27485942]
 [ 0.53837255  0.80983958]
 [ 0.67679705  0.24940991]
 [ 0.30643685  0.26409574]
 [ 0.5804806  -0.05582736]
 [ 0.11467177  0.60923351]
 [ 0.40997223 -0.07701454]
 [ 0.45483563  0.11529686]
 [ 0.82740935 -0.15787864]
 [ 0.5598397   0.56363686]
 [ 0.56464548  0.32748389]
 [ 0.8067377   0.56690393]
 [ 0.390203    0.47284794]
 [ 0.69759607  0.76696337]
 [ 0.75952158  0.47161214]
 [ 0.47074796  0.52394213]
 [ 0.50657147  0.70673497]
 [ 0.73872845  0.75061795]
 [ 0.32729471  0.29900838]
 [ 0.6151503   0.60204261]
 [ 0.32399052  0.10948689]
 [ 0.0790159   0.23530571]
 [ 0.89344804  1.07098428]
 [ 0.38097676  1.02109872]]


In [92]:
def compute_accuracy(y_pred, y_test):
    pred_class = np.argmax(y_pred, axis=1)
    true_class = np.argmax(y_test, axis=1)

    return np.mean(pred_class == true_class)

In [97]:
y_pred = predict(X_test.shape[0], 31, 2, X_test, theta)

accuracy = compute_accuracy(y_pred, y_test)

print(f"accuracy: {(accuracy * 100):.2f}%")

accuracy: 96.60%
